# GRASP: Mortality Prediction on MIMIC-III (With code_mapping)

This notebook runs the GRASP model for mortality prediction **with** `code_mapping` enabled.
Raw codes are mapped to grouped vocabularies before building the embedding table:
- ICD9CM → CCSCM (diagnosis codes → CCS categories)
- ICD9PROC → CCSPROC (procedure codes → CCS categories)
- NDC → ATC (drug codes → ATC categories)

**Paper**: Liantao Ma et al. "GRASP: Generic Framework for Health Status Representation Learning Based on Incorporating Knowledge from Similar Patients." AAAI 2021.

GRASP encodes patient sequences with a backbone (ConCare, GRU, or LSTM), clusters patients via k-means, refines cluster representations with a 2-layer GCN, and blends cluster-level knowledge back into individual patient representations via a learned gating mechanism.

**Model:** GRASP (GRU backbone + GCN cluster refinement)  
**Task:** In-hospital mortality prediction  
**Dataset:** Synthetic MIMIC-III (`dev=False`)

## Step 1: Load the MIMIC-III Dataset

We load the MIMIC-III dataset using PyHealth's `MIMIC3Dataset` class. We use the synthetic dataset hosted on GCS, which requires no credentials.

- `root`: URL to the synthetic MIMIC-III data
- `tables`: Clinical tables to load (diagnoses, procedures, prescriptions)
- `dev`: Set to `False` for the full dataset

In [1]:
!pip install --user --force-reinstall --no-deps /home/lolowo2/git/PyHealth_full_pipeline
!pip install --user ipywidgets

Processing /home/lolowo2/git/PyHealth_full_pipeline
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyhealth: filename=pyhealth-2.0.0-py3-none-any.whl size=602972 sha256=b80f6a6f8692914913c8d955aaf3e37bc5b227b1478bd9002a9faf130aa92dac
  Stored in directory: /tmp/pip-ephem-wheel-cache-9z0d50r7/wheels/a8/55/b7/a62685e2c4f1fab8fd3203610776bd74fee9d3b37834ede4f0
Successfully built pyhealth
  Attempting uninstall: pyhealth
    Found existing installation: pyhealth 2.0.0
    Uninstalling pyhealth-2.0.0:
      Successfully uninstalled pyhealth-2.0.0


In [2]:
import tempfile

from pyhealth.datasets import MIMIC3Dataset

base_dataset = MIMIC3Dataset(
    root="/home/lolowo2",
    tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
    cache_dir=tempfile.TemporaryDirectory().name,
    dev=False,
)

base_dataset.stats()

No config path provided, using default config
Initializing mimic3 dataset from /home/lolowo2 (dev mode: False)
Using provided cache_dir: /tmp/tmpcxipj_37/4f338cfd-b388-50e8-9d9c-fa4872e51b6c
No cached event dataframe found. Creating: /tmp/tmpcxipj_37/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/global_event_df.parquet
Scanning table: patients from /home/lolowo2/PATIENTS.csv.gz
Scanning table: admissions from /home/lolowo2/ADMISSIONS.csv.gz
Scanning table: icustays from /home/lolowo2/ICUSTAYS.csv.gz
Scanning table: diagnoses_icd from /home/lolowo2/DIAGNOSES_ICD.csv.gz
Joining with table: /home/lolowo2/ADMISSIONS.csv.gz
Scanning table: procedures_icd from /home/lolowo2/PROCEDURES_ICD.csv.gz
Joining with table: /home/lolowo2/ADMISSIONS.csv.gz
Scanning table: prescriptions from /home/lolowo2/PRESCRIPTIONS.csv.gz
Joining with table: /home/lolowo2/ADMISSIONS.csv.gz
Caching event dataframe to /tmp/tmpcxipj_37/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/global_event_df.parquet...
Dataset: mimic3
Dev mode: Fa

## Step 2: Define the Mortality Prediction Task

The `MortalityPredictionMIMIC3` task extracts samples from the raw EHR data:
- Extracts diagnosis codes (ICD-9), procedure codes, and drug information from each visit
- Creates binary labels based on in-hospital mortality
- Filters out visits without sufficient clinical codes

We override the task's `input_schema` to enable `code_mapping` on each sequence feature.
This is the **only difference** from the baseline notebook.

In [3]:
from pyhealth.tasks import MortalityPredictionMIMIC3

task = MortalityPredictionMIMIC3()

# Enable code_mapping to collapse granular codes into grouped vocabularies
task.input_schema = {
    "conditions": ("sequence", {"code_mapping": ("ICD9CM", "CCSCM")}),
    "procedures": ("sequence", {"code_mapping": ("ICD9PROC", "CCSPROC")}),
    "drugs": ("sequence", {"code_mapping": ("NDC", "ATC")}),
}

samples = base_dataset.set_task(task)

print(f"Generated {len(samples)} samples")
print(f"\nInput schema: {samples.input_schema}")
print(f"Output schema: {samples.output_schema}")

Setting task MortalityPredictionMIMIC3 for mimic3 base dataset...
Task cache paths: task_df=/tmp/tmpcxipj_37/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/tasks/MortalityPredictionMIMIC3_c67969dc-13b3-5ab7-977f-60956867cc5d/task_df.ld, samples=/tmp/tmpcxipj_37/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/tasks/MortalityPredictionMIMIC3_c67969dc-13b3-5ab7-977f-60956867cc5d/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 46520 patients. (Polars threads: 16)


  0%|          | 0/46520 [00:00<?, ?it/s]

Rank 0 inferred the following `['bytes']` data format.


100%|██████████| 46520/46520 [01:03<00:00, 734.65it/s]

Worker 0 finished processing patients.
Fitting processors on the dataset...


...

  0%|          | 0/9583 [00:00<?, ?it/s]

Rank 0 inferred the following `['str', 'str', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:1']` data format.


100%|██████████| 9583/9583 [00:02<00:00, 3953.17it/s]

Worker 0 finished processing samples.


Cached processed samples to /tmp/tmpcxipj_37/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/tasks/MortalityPredictionMIMIC3_c67969dc-13b3-5ab7-977f-60956867cc5d/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Generated 9583 samples

Input schema: {'conditions': ('sequence', {'code_mapping': ('ICD9CM', 'CCSCM')}), 'procedures': ('sequence', {'code_mapping': ('ICD9PROC', 'CCSPROC')}), 'drugs': ('sequence', {'code_mapping': ('NDC', 'ATC')})}
Output schema: {'mortality': 'binary'}


## Step 3: Dataset Statistics

Each sample represents one hospital visit with:
- **conditions**: List of mapped CCS diagnosis categories (collapsed from ICD-9)
- **procedures**: List of mapped CCS procedure categories (collapsed from ICD-9 PROC)
- **drugs**: List of drug codes (NDC → ATC mapping attempted, falls back to raw if no match)
- **mortality**: Binary label (0 = survived, 1 = deceased)

In [4]:
print("Sample structure:")
print(samples[0])

print("\n" + "=" * 50)
print("Processor Vocabulary Sizes:")
print("=" * 50)
for key, proc in samples.input_processors.items():
    if hasattr(proc, 'code_vocab'):
        print(f"{key}: {len(proc.code_vocab)} codes (including <pad>, <unk>)")

mortality_count = sum(float(s.get("mortality", 0)) for s in samples)
print(f"\nTotal samples: {len(samples)}")
print(f"Mortality rate: {mortality_count / len(samples) * 100:.2f}%")
print(f"Positive samples: {int(mortality_count)}")
print(f"Negative samples: {len(samples) - int(mortality_count)}")

Sample structure:
{'hadm_id': '164713', 'patient_id': '10004', 'conditions': tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16]), 'procedures': tensor([2, 3, 4, 5, 6, 7, 8]), 'drugs': tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 10, 11, 11, 11, 12, 12, 13, 14, 11,
        15, 11, 16, 17, 17, 11, 18,  2,  3, 19, 20, 21, 22, 11, 20, 21, 22, 20,
        21, 22, 11, 12, 11, 23, 24, 25, 26, 27, 28, 11, 29, 30, 31, 32, 14, 24,
        25, 33, 26, 34, 34, 34, 35, 29, 30, 32, 36, 37, 38, 39, 38, 39, 34, 38,
        39, 40, 41,  9, 14, 14,  7,  5,  6,  9,  9,  8,  9,  9, 42, 14, 14,  5,
         6, 11,  2,  3, 17, 17,  9,  9, 28, 28, 27, 43, 44, 13, 45, 46, 11,  9,
         4, 28,  3, 23, 24, 25, 26, 24, 25, 33, 26, 24, 25, 33, 26, 24, 25, 26,
        47, 46, 19, 14, 46, 48, 49, 14, 46, 50, 51, 48,  5,  6, 14, 49, 52, 49,
        49,  7,  9, 53, 52, 49, 49, 14, 54, 11, 50, 51, 14, 55, 50, 51, 12, 55,
        56, 38, 39, 50, 51, 55, 11, 16, 12, 50, 51, 11, 57, 46, 14,  7, 50, 51,

## Step 4: Split the Dataset

We split the data by patient to avoid data leakage — all visits from a given patient go into the same split.

In [5]:
from pyhealth.datasets import split_by_patient

train_dataset, val_dataset, test_dataset = split_by_patient(
    samples, [0.8, 0.1, 0.1], seed=42
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Training samples: 7712
Validation samples: 909
Test samples: 962


## Step 5: Create Data Loaders

Data loaders batch the samples and handle data feeding during training and evaluation.

In [6]:
from pyhealth.datasets import get_dataloader

train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)

print(f"Training batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")
print(f"Test batches: {len(test_dataloader)}")

Training batches: 241
Validation batches: 29
Test batches: 31


## Step 6: Initialize the GRASP Model

The GRASP model automatically handles different feature types via `EmbeddingModel`.
Sequence features (diagnosis/procedure/drug codes) are embedded using learned embeddings,
and each feature gets its own `GRASPLayer`.

### Key Parameters:
- `embedding_dim`: Dimension of code embeddings (default: 128)
- `hidden_dim`: Hidden dimension of the backbone (default: 128)
- `cluster_num`: Number of patient clusters for knowledge sharing (default: 2)
- `block`: Backbone encoder — `"ConCare"`, `"GRU"`, or `"LSTM"` (default: `"ConCare"`)
- `dropout`: Dropout rate for regularization (default: 0.5)

In [7]:
from pyhealth.models import GRASP

model = GRASP(
    dataset=samples,
      embedding_dim=64,
      hidden_dim=64,
      cluster_num=4,
      block="GRU",
      dropout=0.5,
)

print(f"Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"\nModel architecture:")
print(model)

Model initialized with 213,895 parameters

Model architecture:
GRASP(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(268, 64, padding_idx=0)
    (procedures): Embedding(204, 64, padding_idx=0)
    (drugs): Embedding(1295, 64, padding_idx=0)
  ))
  (grasp): ModuleDict(
    (conditions): GRASPLayer(
      (backbone): RNNLayer(
        (dropout_layer): Dropout(p=0, inplace=False)
        (rnn): GRU(64, 64, batch_first=True)
      )
      (relu): ReLU()
      (tanh): Tanh()
      (sigmoid): Sigmoid()
      (dropout): Dropout(p=0.5, inplace=False)
      (weight1): Linear(in_features=64, out_features=1, bias=True)
      (weight2): Linear(in_features=64, out_features=1, bias=True)
      (GCN): GraphConvolution()
      (GCN_2): GraphConvolution()
      (bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (procedures): GRASPLayer(
      (backbone): RNNLayer(
        (dropout_layer): Dropout(p=0, inplace=Fal

## Step 7: Train the Model

We use PyHealth's `Trainer` class which handles:
- Training loop with automatic batching
- Validation during training
- Model checkpointing based on validation metrics

We monitor the **ROC-AUC** score on the validation set.

In [9]:
from pyhealth.trainer import Trainer

trainer = Trainer(
    model=model,
    metrics=["roc_auc", "pr_auc", "accuracy", "f1"],
)

trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=50,
    monitor="pr_auc",
    optimizer_params={"lr": 5e-4},
)

GRASP(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(268, 64, padding_idx=0)
    (procedures): Embedding(204, 64, padding_idx=0)
    (drugs): Embedding(1295, 64, padding_idx=0)
  ))
  (grasp): ModuleDict(
    (conditions): GRASPLayer(
      (backbone): RNNLayer(
        (dropout_layer): Dropout(p=0, inplace=False)
        (rnn): GRU(64, 64, batch_first=True)
      )
      (relu): ReLU()
      (tanh): Tanh()
      (sigmoid): Sigmoid()
      (dropout): Dropout(p=0.5, inplace=False)
      (weight1): Linear(in_features=64, out_features=1, bias=True)
      (weight2): Linear(in_features=64, out_features=1, bias=True)
      (GCN): GraphConvolution()
      (GCN_2): GraphConvolution()
      (bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (procedures): GRASPLayer(
      (backbone): RNNLayer(
        (dropout_layer): Dropout(p=0, inplace=False)
        (rnn): GRU(64, 64, batch_first=True)
      )
      

Epoch 0 / 50:   0%|          | 0/241 [00:00<?, ?it/s]

--- Train epoch-0, step-241 ---
loss: 0.0107


Evaluation: 100%|██████████| 29/29 [00:00<00:00, 86.37it/s]

--- Eval epoch-0, step-241 ---
roc_auc: 0.5447
pr_auc: 0.1213
accuracy: 0.8152
f1: 0.0667
loss: 1.4196
New best pr_auc score (0.1213) at epoch-0, step-241



...

--- Eval epoch-48, step-11809 ---
roc_auc: 0.5248
pr_auc: 0.1179
accuracy: 0.8361
f1: 0.0629
loss: 1.9460



Epoch 49 / 50:   0%|          | 0/241 [00:00<?, ?it/s]

--- Train epoch-49, step-12050 ---
loss: 0.0025


Evaluation: 100%|██████████| 29/29 [00:00<00:00, 84.44it/s]

--- Eval epoch-49, step-12050 ---
roc_auc: 0.5306
pr_auc: 0.1197
accuracy: 0.8185
f1: 0.0782
loss: 1.8566
Loaded best model


## Step 8: Evaluate on Test Set

After training, we evaluate the model on the held-out test set to measure its generalization performance.

In [10]:
test_results = trainer.evaluate(test_dataloader)

print("\n" + "=" * 50)
print("Test Set Performance (WITH code_mapping)")
print("=" * 50)
for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")

Evaluation: 100%|██████████| 31/31 [00:00<00:00, 93.73it/s]


Test Set Performance (WITH code_mapping)
roc_auc: 0.5783
pr_auc: 0.1768
accuracy: 0.8534
f1: 0.1557
loss: 1.8237


## Step 9: Extract Patient Embeddings

GRASP produces patient embeddings that encode health status enriched with knowledge from similar patients.
These embeddings can be used for downstream tasks like patient similarity search, cohort discovery, or transfer learning.

In [11]:
import torch

model.eval()
test_batch = next(iter(test_dataloader))
test_batch["embed"] = True

with torch.no_grad():
    output = model(**test_batch)

print(f"Embedding shape: {output['embed'].shape}")
print(f"  - Batch size: {output['embed'].shape[0]}")
print(f"  - Embedding dim: {output['embed'].shape[1]}")

print("\n" + "=" * 50)
print("Sample Predictions:")
print("=" * 50)
predictions = output["y_prob"].cpu().numpy()
true_labels = output["y_true"].cpu().numpy()

for i in range(min(5, len(predictions))):
    pred = predictions[i][0]
    true = int(true_labels[i][0])
    print(f"Patient {i + 1}: Predicted={pred:.3f}, True={true}, Prediction={'Mortality' if pred > 0.5 else 'Survival'}")

Embedding shape: torch.Size([32, 192])
  - Batch size: 32
  - Embedding dim: 192

Sample Predictions:
Patient 1: Predicted=0.000, True=0, Prediction=Survival
Patient 2: Predicted=0.000, True=0, Prediction=Survival
Patient 3: Predicted=0.948, True=0, Prediction=Mortality
Patient 4: Predicted=0.000, True=0, Prediction=Survival
Patient 5: Predicted=0.000, True=1, Prediction=Survival
